# 02 · Data preparation and versioning

Preserve source evidence and make each transformation explicit. No imputation, target resampling or outlier removal is justified by the initial checks.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'retainwise').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from retainwise.data import RAW, PROCESSED, DATA, validate_source, prepare, dictionary, write_version, load_data
from retainwise.analysis import rate_table, distribution, hypothesis_evidence, HYPOTHESES
pd.set_option('display.max_columns', 40)


In [2]:
raw = pd.read_csv(RAW)
display(pd.Series(validate_source(raw)))
assert raw.EmployeeNumber.is_unique
assert raw[['EmployeeCount', 'Over18', 'StandardHours']].nunique().eq(1).all()

rows                                                   1470
columns                                                  35
missing_values                                            0
duplicate_rows                                            0
identifier_unique                                      True
constant_columns     [EmployeeCount, Over18, StandardHours]
target_counts                      {'No': 1233, 'Yes': 237}
dtype: object

## Transformation decisions

Remove EmployeeNumber (unnecessary identifier) and the verified constants EmployeeCount, Over18 and StandardHours. Retain Attrition; derive Attrition_Flag and readable Status. AgeBand uses closed integer intervals 18–29, 30–39, 40–49, 50–60. TenureBand uses 0–1, 2–5, 6–10, 11+ years. The implementation rejects ages outside its reviewed bands. Removing identifiers does not establish anonymisation.

In [3]:
clean = prepare(raw)
PROCESSED.mkdir(parents=True, exist_ok=True)
clean.to_csv(PROCESSED / 'employee_attrition_v1.csv', index=False, lineterminator='\n')
dictionary(clean).to_csv(PROCESSED / 'data_dictionary_v1.csv', index=False, lineterminator='\n')
clean, metadata = write_version()
assert len(clean) == len(raw)
assert clean.Attrition_Flag.eq(raw.Attrition.map({'Yes': 1, 'No': 0})).all()
assert not clean[['AgeBand', 'TenureBand']].isna().any().any()
display(pd.Series(metadata))
display(dictionary(clean))

version                                                            v1
source              IBM HR Analytics Employee Attrition & Performance
source_url          https://www.kaggle.com/datasets/pavansubhasht/...
source_sha256       a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f...
processed_sha256    cf8204eeefb2adfe42e16ef31c58a54e4abc99309f5aa9...
validation          {'rows': 1470, 'columns': 35, 'missing_values'...
rows                                                             1470
columns                                                            35
excluded            [EmployeeNumber, EmployeeCount, Over18, Standa...
transformations     [Attrition_Flag: Yes=1, No=0, Status: Yes=Attr...
licence             Database: Open Database, Contents: Database Co...
limitations         [Fictional population, No dated exits, No caus...
dtype: object

,field,dtype,description,unique_values,missing
0,Age,int64,Age in years; used only for aggregate audit.,43,0
1,Attrition,object,Source outcome: Yes / No; no exit date is supp...,2,0
2,BusinessTravel,object,Source travel-frequency category; no measured ...,3,0
3,DailyRate,int64,Source daily rate; units and interpretation no...,886,0
4,Department,object,Source organisational department.,3,0
5,DistanceFromHome,int64,Source distance from home; distance unit not d...,29,0
6,Education,int64,Source ordinal scale: 1=Below college; 2=Colle...,5,0
7,EducationField,object,Source education field category.,6,0
8,EnvironmentSatisfaction,int64,Source ordinal scale: 1=Low; 2=Medium; 3=High;...,4,0
9,Gender,object,Source-labelled Female/Male; incomplete repres...,2,0


## Version contract

The dedicated `data/processed/v1` folder contains the dataset, dictionary and metadata with SHA-256 hashes. Repeated execution is deterministic. A different source population or revised transformation requires a reviewed v2, preserving v1. Original columns and outcome balance remain unchanged except for the four documented exclusions.